# Chargement & Nettoyage
## Arene ML - Prediction de souscription bancaire (Bank Marketing UCI)


Charger le dataset Bank Marketing depuis UCI, explorer, nettoyer et livrer un DataFrame pret pour Mathieu (Arena) et Amos (WebApp).



## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

pd.set_option('display.max_columns', 30)
print('Imports OK')


## 1. Chargement du dataset (UCI)

On utilise la base officielle UCI Bank Marketing (id=222).  
- **45 211 clients** contactes par une banque portugaise  
- **16 features** (mix numerique + categoriel)  
- **1 cible** : `y` -- `yes` / `no` (souscription au depot a terme)


In [ ]:
bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

print('Metadata :')
print(bank_marketing.metadata)
print('\nVariable information :')
print(bank_marketing.variables)


In [ ]:
print(f'Forme de X : {X.shape}')
print(f'Forme de y : {y.shape}')
print('\nPremiere ligne :')
X.head()


## 2. Exploration initiale (EDA)

In [ ]:
df = X.copy()
df['y'] = y['y']

print('=== Types techniques (pandas) ===')
print(df.dtypes)
print(f'\nForme : {df.shape}')


In [ ]:
print('=== Statistiques descriptives ===')
df.describe(include='all').T


In [ ]:
# Distribution de la cible -- desequilibre de classes
counts = df['y'].value_counts()
pct = df['y'].value_counts(normalize=True) * 100
summary = pd.DataFrame({'count': counts, 'pct (%)': pct.round(1)})
print('=== Distribution de la cible ===')
print(summary)

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(summary.index, summary['count'], color=['#e05c5c', '#5ca0e0'])
ax.set_title('Distribution de la cible y (yes / no)')
ax.set_ylabel('Nombre de clients')
for bar, (_, row) in zip(ax.patches, summary.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 200,
            f'{row["pct (%)"]:.1f}%',
            ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=120)
plt.show()
print('ATTENTION : 88% no / 12% yes -- metrique retenue : ROC-AUC + recall(yes)')


## 3. Valeurs manquantes



**Particularite UCI Bank Marketing :** la valeur `unknown` apparait dans `job`, `education`, `contact`, `poutcome`. Selon UCI, c'est une **categorie officielle** (pas un NaN). On la detecte explicitement.


In [ ]:
# NaN classiques
manquants = df.isna().sum()
pourcentage = (df.isna().mean() * 100).round(1)
resume = pd.DataFrame({'manquants': manquants, 'pct (%)': pourcentage})
print('=== NaN classiques ===')
nan_df = resume[resume['manquants'] > 0]
if nan_df.empty:
    print('Aucun NaN detecte -- le dataset UCI est propre sur ce point.')
else:
    print(nan_df.sort_values('pct (%)', ascending=False))


In [ ]:
# Valeurs 'unknown' par colonne
cols_with_unknown = [
    c for c in df.columns
    if df[c].dtype == object and (df[c].str.lower() == 'unknown').any()
]
print('=== Colonnes contenant unknown ===')
for col in cols_with_unknown:
    n = (df[col].str.lower() == 'unknown').sum()
    pct = n / len(df) * 100
    print(f'  {col:15s} -> {n:5d} unknowns ({pct:.1f}%)')

print()
print('Decision retenue : unknown = categorie officielle UCI, conserve tel quel.')
print('Option alternative : remplacer par le mode -- a comparer via cross-val si besoin.')


## 4. Valeurs aberrantes (outliers)


On inspecte les colonnes numeriques les plus exposees.


In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Colonnes numeriques :', num_cols)

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
for i, col in enumerate(num_cols[:8]):
    axes[i].boxplot(df[col].dropna(), vert=False)
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(labelsize=7)
plt.suptitle('Boxplots -- detection outliers (regle IQR visuelle)', fontsize=11)
plt.tight_layout()
plt.savefig('boxplots_outliers.png', dpi=120)
plt.show()


In [ ]:
# Quantification IQR sur les variables les plus exposees
for col in ['balance', 'duration', 'campaign']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    basse = Q1 - 1.5 * IQR
    haute = Q3 + 1.5 * IQR
    n_out = ((df[col] < basse) | (df[col] > haute)).sum()
    print(f'{col:12s} -> bornes [{basse:.0f}, {haute:.0f}] | outliers : {n_out} ({n_out/len(df)*100:.1f}%)')

print()
print('Decision : outliers conserves -- valeurs reelles (comptes negatifs, campagnes intenses).')
print('duration : conservee pour comparatif mais inutilisable en production.')


## 5. Le cas `duration` -- fuite de donnees en production

> La duree du dernier appel est ultra-predictive mais on ne la connait pas avant d'appeler.  
> On cree deux versions du dataset pour que Mathieu puisse comparer l'impact.


In [ ]:
df_corr = df.copy()
df_corr['y_bin'] = (df_corr['y'] == 'yes').astype(int)
corr_val = df_corr['duration'].corr(df_corr['y_bin'])
print(f'Correlation Pearson duration / y_bin : {corr_val:.3f}')
print()
print('-> 2 versions de X seront exportees :')
print('   X_full   : toutes les features (avec duration)')
print('   X_no_dur : sans duration (version production realiste)')
